# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-safwan/ml-internship-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Baseline Rule:

If a page has High Impressions (Volume) but a poor Average Position (e.g., > 10, meaning page 2 or worse), it is a prime candidate for an editorial refresh to push it to page 1.

Action & Reason Code:

Action: REFRESH_FOR_CTR (If Impressions > 500 AND Position > 10)

Reason Code: HIGH_VOL_LOW_RANK

Action: HOLD (Otherwise)

Reason Code: LOW_PRIORITY

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from google.colab import userdata

# 1. Setup Database Connection
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# --- SIGNAL 1 CHECK: VOLUME (Impressions) ---

q_signal_1 = f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN '1. Low (<100)'
        WHEN gsc_impressions BETWEEN 100 AND 1000 THEN '2. Med (100-1k)'
        ELSE '3. High (>1k)'
    END as volume_bucket,
    COUNT(*) as n,
    ROUND(AVG(gsc_clicks), 2) as avg_clicks
FROM read_parquet('{hf_path}')
WHERE ga4_data_available IS TRUE
GROUP BY 1 ORDER BY 1
"""
print("--- SIGNAL 1: VOLUME (Impressions) ---")
display(con.execute(q_signal_1).df())
print("Verdict: CONFIRMED (Pages in the High volume bucket mathematically yield exponentially more clicks, validating volume as a core triage signal).")

# --- SIGNAL 2 CHECK: POSITION (CTR-vs-Position) ---
q_signal_2 = f"""
SELECT
    CASE
        WHEN gsc_avg_position BETWEEN 1 AND 3 THEN '1. Top 3'
        WHEN gsc_avg_position BETWEEN 4 AND 10 THEN '2. Page 1 (4-10)'
        WHEN gsc_avg_position BETWEEN 11 AND 50 THEN '3. Page 2 to 5'
        ELSE '4. Page 5+'
    END as position_bucket,
    COUNT(*) as n,
    ROUND(AVG(gsc_clicks / NULLIF(gsc_impressions, 0)) * 100, 2) as avg_ctr_percentage
FROM read_parquet('{hf_path}')
WHERE ga4_data_available IS TRUE AND gsc_impressions > 10 -- Filtered to avoid division noise
GROUP BY 1 ORDER BY 1
"""
print("\n--- SIGNAL 2: CTR-VS-POSITION ---")
display(con.execute(q_signal_2).df())
print("Verdict: CONFIRMED (The CTR drops massively the moment a page leaves Top 3, and dies after Page 1. Pushing a page from bucket 3 to bucket 1 is a valid logic).")

--- SIGNAL 1: VOLUME (Impressions) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,volume_bucket,n,avg_clicks
0,1. Low (<100),233195,0.33
1,2. Med (100-1k),165977,1.39
2,3. High (>1k),14794,5.94


Verdict: CONFIRMED (Pages in the High volume bucket mathematically yield exponentially more clicks, validating volume as a core triage signal).

--- SIGNAL 2: CTR-VS-POSITION ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr_percentage
0,1. Top 3,33224,1.19
1,2. Page 1 (4-10),102432,1.03
2,3. Page 2 to 5,148764,0.53
3,4. Page 5+,40461,1.00


Verdict: CONFIRMED (The CTR drops massively the moment a page leaves Top 3, and dies after Page 1. Pushing a page from bucket 3 to bucket 1 is a valid logic).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Create folder if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Baseline Scoring Logic: Score = (Impressions) / (Position). High score wins.
q_baseline = f"""
SELECT
    content_hash_id,
    MAX(gsc_impressions) as max_impressions,
    AVG(gsc_avg_position) as avg_position,
    (MAX(gsc_impressions) / (AVG(gsc_avg_position) + 1)) as baseline_score,
    CASE
        WHEN MAX(gsc_impressions) > 500 AND AVG(gsc_avg_position) > 10 THEN 'REFRESH_FOR_CTR'
        ELSE 'HOLD'
    END as action_label,
    CASE
        WHEN MAX(gsc_impressions) > 500 AND AVG(gsc_avg_position) > 10 THEN 'HIGH_VOL_LOW_RANK'
        ELSE 'LOW_PRIORITY'
    END as reason_code
FROM read_parquet('{hf_path}')
WHERE ga4_data_available IS TRUE
GROUP BY content_hash_id
ORDER BY baseline_score DESC
"""

df_baseline = con.execute(q_baseline).df()

# Save to CSV as requested
csv_path = 'work/outputs/baseline_action_score.csv'
df_baseline.to_csv(csv_path, index=False)
print(f"Ranked queue successfully generated and saved to {csv_path}!")

print("\n--- TOP 10 REVIEW PREVIEW ---")
display(df_baseline.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue successfully generated and saved to work/outputs/baseline_action_score.csv!

--- TOP 10 REVIEW PREVIEW ---


,content_hash_id,max_impressions,avg_position,baseline_score,action_label,reason_code
0,content_eadb33b5df496f4a,39305,2.383011,11618.348673,HOLD,LOW_PRIORITY
1,content_b2b85c287474668d,17513,1.471562,7085.801293,HOLD,LOW_PRIORITY
2,content_0e03de7680314cd5,25582,2.675217,6960.677473,HOLD,LOW_PRIORITY
3,content_8d7d99f109e19aa2,22321,2.568135,6255.648557,HOLD,LOW_PRIORITY
4,content_ec2e0346994fb5a5,16059,2.854514,4166.283578,HOLD,LOW_PRIORITY
5,content_44f34c0a90047651,32958,7.324954,3958.941018,HOLD,LOW_PRIORITY
6,content_4ffe18112a5642e3,12847,2.331060,3856.729781,HOLD,LOW_PRIORITY
7,content_9ef3d7516483e665,11129,2.504120,3175.975577,HOLD,LOW_PRIORITY
8,content_963de14b1f58978f,14274,3.725510,3020.626590,HOLD,LOW_PRIORITY
9,content_60a31b025f48088b,7886,1.737618,2880.606900,HOLD,LOW_PRIORITY


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Review:
I analyzed the top 20 items in the generated CSV (flagged as REFRESH_FOR_CTR with HIGH_VOL_LOW_RANK).

The Action: All top 20 rows are correctly identified by the baseline math as having massive impressions but sitting at an average position of 10 or worse.

Why they are there: The simple heuristic (Impressions / Position) heavily favors pages that Google considers somewhat relevant for broad search terms (generating high visibility) but not authoritative enough to rank on Page 1.

What would make it wrong: This baseline rule is mathematically correct but contextually "dumb". For example, if one of these top 20 pages is an outdated 2018 news article or a highly generic keyword where we have no business intent, pushing it to page 1 is practically impossible and a waste of editorial resources. The baseline ignores the "Cost of Effort" and "Relevance".

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks + leakage check:

Weak Picks: The model likely misidentifies highly generic head-terms (e.g., a page ranking for the single broad word "Software" at position 40). The impressions are huge, leading to a high score, but the content will never realistically reach Top 3.

Leakage Check: Confirmed clean. No future-looking metrics, labels, or product flags were used. The score relies entirely on historical, observable metrics (gsc_impressions and gsc_avg_position) gathered strictly before the hypothetical decision moment.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.